# Assignment 7
**UCS420 – Cognitive Computing**

**NLP - I**

**Name:** Sumit Bansal  
**Roll No:** 1024170041


In [ ]:
import nltk
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from collections import Counter
from nltk.util import ngrams
from nltk import pos_tag
import string

# Downloading the necessary NLTK datasets so it doesn't throw errors
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')


## Dataset Generation (Starter Code)
*Note: Just run the starter notebook code `generate_dataset()` and `process_ticket()` here. The code below assumes we have a `tickets` list ready.*


In [ ]:
# Paste your `generate_dataset()` and `process_ticket()` functions from the starter notebook here.
# tickets = generate_dataset('1024170041')

# ---------------------------------------------------------
# MOCK DATA FOR TESTING - Replace this part with the actual generated tickets
# ---------------------------------------------------------
TEST_LINES = [
    "Please email support@domain.com about the Wi-Fi issue. It isn't working on version 2.0.1.",
    "Call us at +1-800-555-0199 or visit https://help.domain.com. Confidence is 95.5%."
]
tickets = TEST_LINES.copy()


## Q.1 Preprocessing


In [ ]:
# 1. Preprocessing
# Running process_ticket() style logic to get all the counts.

def compute_stats(ticket_text):
    sentences = nltk.sent_tokenize(ticket_text)
    tokens = word_tokenize(ticket_text)
    punct_tokens = [t for t in tokens if t in string.punctuation]
    stop_words = set(stopwords.words('english'))
    
    # Filtering out punctuation and stopwords for the final tokens
    final_tokens = [t.lower() for t in tokens if t not in string.punctuation and t.lower() not in stop_words]
    
    return {
        "num_sentences": len(sentences),
        "num_tokens": len(tokens),
        "num_punctuation": len(punct_tokens),
        "num_stopwords": len([t for t in tokens if t.lower() in stop_words]),
        "num_unique_tokens": len(set(tokens)),
        "num_final_tokens": len(final_tokens),
        "actual_tokens": tokens,
        "actual_punct": punct_tokens,
        "actual_stopwords": [t for t in tokens if t.lower() in stop_words],
        "actual_final": final_tokens
    }

# Process all tickets
stats_list = [compute_stats(t) for t in tickets]

# Making a nice pandas dataframe to show the table
df_stats = pd.DataFrame(stats_list)
cols_to_sum = ['num_sentences', 'num_tokens', 'num_punctuation', 'num_stopwords', 'num_unique_tokens', 'num_final_tokens']

# Adding a row at the bottom for total counts
total_row = df_stats[cols_to_sum].sum()
total_row.name = 'TOTAL'
df_display = pd.concat([df_stats[cols_to_sum], total_row.to_frame().T])

print("--- Ticket Statistics Table ---")
display(df_display)

print("\n--- Output for one selected ticket ---")
selected = stats_list[0]
print("Tokens:", selected["actual_tokens"])
print("Punctuation:", selected["actual_punct"])
print("Stopwords:", selected["actual_stopwords"])
print("Final Tokens:", selected["actual_final"])


## Q.2 Regex extraction and replacement


In [ ]:
# 2. Regex Extraction and Replacement

# Patterns for extracting specific things
extract_patterns = {
    "words_longer_than_5": r'\b[a-zA-Z]{6,}\b',
    "numbers_with_decimals_versions": r'\b\d+(?:\.\d+)+\b|\b\d+\b',
    "capitalized_words": r'\b[A-Z][a-zA-Z]*\b',
    "words_beginning_with_vowel": r'\b[AEIOUaeiou][a-zA-Z]*\b'
}

# Patterns to replace personal/link info with tags
replace_patterns = [
    (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '<EMAIL>'),
    (r'https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+', '<URL>'),
    (r'(?:\+\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '<PHONE>')
]

for i, line in enumerate(TEST_LINES):
    print(f"\n--- Checking Line {i+1} ---")
    print("Original Text:", line)
    
    # Testing extractions
    print("\nExtractions:")
    for name, pat in extract_patterns.items():
        matches = re.findall(pat, line)
        print(f"{name:35s}: {matches}")
        
    # Testing replacements
    subbed_text = line
    for pat, repl in replace_patterns:
        subbed_text = re.sub(pat, repl, subbed_text)
    print("\nReplaced Text:", subbed_text)


## Q.3 Custom tokenization


In [ ]:
# 3. Custom Tokenization
# Using one big regex pattern to catch URLs, Emails, Phone numbers, Decimals, Versions, Hyphenated words, etc.
custom_pattern = r"https?://\S+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|(?:\+\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}|\b\d+(?:\.\d+)+\b|\b[A-Za-z]+(?:[-'][A-Za-z]+)+\b|\b\w+\b|\S"

tokenizer = RegexpTokenizer(custom_pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)

# Let's compare my custom tokenizer with NLTK's standard word_tokenize
sample_text = "Please email support@domain.com about the Wi-Fi issue. It isn't working on version 2.0.1."
print("NLTK word_tokenize:\n", word_tokenize(sample_text))
print("\nmy_tokenize:\n", my_tokenize(sample_text))


## Q.4 Stemming and lemmatization


In [ ]:
# 4. Stemming and Lemmatization

# Grabbing the unique final tokens from Q1
unique_final_tokens = list(set([t for stat in stats_list for t in stat['actual_final']]))

port = PorterStemmer()
lanc = LancasterStemmer()

# For Regexp Stemmer, 'ing', 's', and 'ed' are super common in IT tickets (like 'working', 'crashed', 'issues')
regex_stem = RegexpStemmer('ing$|s$|ed$', min=4) 
lemmatizer = WordNetLemmatizer()

# Helper to get the right Part-of-Speech tag so the lemmatizer works better
from nltk.corpus import wordnet
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

diff_list = []
for word in unique_final_tokens:
    p_stem = port.stem(word)
    l_stem = lanc.stem(word)
    r_stem = regex_stem.stem(word)
    lem = lemmatizer.lemmatize(word, get_wordnet_pos(word))
    
    # We only care about words where the tools give different results
    if not (p_stem == l_stem == r_stem == lem):
        diff_list.append({
            "Word": word,
            "Porter": p_stem,
            "Lancaster": l_stem,
            "Regexp": r_stem,
            "Lemmatized": lem
        })

df_stem_diff = pd.DataFrame(diff_list)
print("Words with differing stemming/lemmatization outputs:")
display(df_stem_diff)


## Q.5 Corpus statistics


In [ ]:
# 5. Corpus Statistics

# Building the sequences
S1 = [] 
for t in tickets: S1.extend(word_tokenize(t))

S2 = [w for w in S1 if w not in string.punctuation] 

stop_words = set(stopwords.words('english'))
S3 = [w.lower() for w in S2 if w.lower() not in stop_words]
S4 = [port.stem(w) for w in S3]

# Simple function to calculate total, unique, and TTR
def calc_stats(tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total > 0 else 0
    return total, unique, ttr

stats_corpus = {
    "S1 (All Tokens)": calc_stats(S1),
    "S2 (No Punctuation)": calc_stats(S2),
    "S3 (Final Tokens)": calc_stats(S3),
    "S4 (Porter Stemmed)": calc_stats(S4)
}

df_corpus = pd.DataFrame.from_dict(stats_corpus, orient='index', columns=['Total Tokens', 'Unique Tokens', 'TTR'])
display(df_corpus)

print("\nTop 10 words in S1:", [word for word, count in Counter(S1).most_common(10)])
print("Top 10 words in S3:", [word for word, count in Counter(S3).most_common(10)])

# Plotting Zipf's Law (log(rank) vs log(freq)) for S3
freqs = [count for word, count in Counter(S3).most_common()]
ranks = range(1, len(freqs) + 1)

if len(freqs) > 0:
    plt.figure(figsize=(6,4))
    plt.plot(np.log(ranks), np.log(freqs), marker='o')
    plt.title("Zipf's Law - log(Rank) vs log(Frequency) for S3")
    plt.xlabel("log(Rank)")
    plt.ylabel("log(Frequency)")
    plt.grid(True)
    plt.show()

# Checking top 5 bigrams (pairs of words)
print("Top 5 bigrams in S1:", Counter(ngrams(S1, 2)).most_common(5))
print("Top 5 bigrams in S3:", Counter(ngrams(S3, 2)).most_common(5))


## Q.6 Interpretation

**(a) Why does TTR change from S1 to S4 in your dataset?**  
As we clean up the text from S1 to S3 (by dropping punctuation and common stopwords), our total token count drops a lot. But we still keep most of the unique, meaningful words. This causes the TTR (Type-Token Ratio) to go up. Then, when we apply Porter stemming in S4, different forms of a word (like 'working' and 'works') get squashed into the same stem ('work'). This reduces the number of unique words, which makes the TTR drop again.

**(b) Give one example where `word_tokenize()` and your tokenizer differ. Explain the difference.**  
Take a word like `"Wi-Fi"`. NLTK's `word_tokenize()` splits it into three pieces `['Wi', '-', 'Fi']` because it treats the hyphen as just punctuation. My custom regex tokenizer, on the other hand, keeps it as a single token `['Wi-Fi']` because I explicitly added a pattern to keep hyphenated words together.

**(c) Give one example where stemming produces an undesirable result.**  
Stemmers can be pretty blind and aggressive. For example, if you pass `"university"` into a Porter stemmer, it just chops the end off and gives you `"univers"`. It doesn't care if the output isn't a real dictionary word, which can make the text look weird and hard to use for other tasks later.

**(d) Give one example where POS-aware lemmatization preserves meaning better than stemming.**  
Let's look at the word `"leaves"`. A basic stemmer will just cut it down to `"leav"`. But a POS-aware lemmatizer actually checks the grammar context. If it's a noun (like leaves on a tree), it intelligently reduces it to `"leaf"`. If it's a verb (like he leaves the room), it keeps it as `"leave"`. It actually understands what the word means in context!
